# Tropical cyclone BUFR data - Decoding 
authors: Marijana Crepulja, Roberto Ribas (ECMWF)

#### This notebook demonstrates how to decode BUFR message using the `eccodes` library.


### Explore a BUFR message with comand bufr_dump

#### bufr_dump -p  bufr_khanun.bufr 

> DataCategory=7 <br>
dataSubCategory=32 <br>
masterTablesVersionNumber=35 <br>
localTablesVersionNumber=0 <br>
unexpandedDescriptors={316082} <br>





 ### Setup enviroment
First, we'll install and import the required libraries.

In [14]:
import sys
from eccodes import *
import numpy as np


 ### Open BUFR file and print number of messages 

In [46]:
tc_filename="bufr_khanun.bufr"
with open(tc_filename,'rb') as f:
    nmsg=codes_count_in_file(f)
    print(f" number of messages {nmsg}")

 number of messages 1



 ### Decode storm identification from the BUFR message

In [47]:
tc_filename="bufr_khanun.bufr"
with open(tc_filename,'rb') as f:
        
    ### we need to get a 'handle' to a BUFR message to be able to read it
    bid=codes_bufr_new_from_file(f) 
    
    ### next we have to unpack the data section to be able to see the values
    codes_set(bid,'unpack',1)
    
    ### at this point we can look for a particular key ( use bufr_dump bufr_khanun.bufr to explore the keys)
    stormName=codes_get(bid,'longStormName')
    stormIdentifier=codes_get(bid,'stormIdentifier')
        
    print(f"storm : {stormName} identifier: {stormIdentifier} ")

storm :     KHANUN identifier: 08W 



 ### Decode storm centre location from the BUFR message

Some keys in the BUFR message have a rank ( surrounded by the hash symbol #) for example **#1#latitude**. To access these keys we need to create a string like this:

In [ ]:
rank=1
key_lat=f"#{rank}#latitude"
print(f" eccodes BUFR key to decode: {key_lat}")

#### Get and print value of the latitude

In [49]:
lat=codes_get(bid,key_lat)
print(f" eccode key {key_lat} value {lat}")

 eccode key #1#latitude value 26.2


#### Get and print values of the longitude

In [50]:
key_lon=f"#{rank}#longitude"
lon=codes_get(bid,key_lon)
print(f" eccode key: {key_lon} value: {lon}")

 eccode key: #1#longitude value: 125.60000000000001



 ### Decode forecasted position and pressure of the tropical cyclone from the BUFR message

The goal of this section is to decode the latitude,longitude and pressue of the storm centre at the forecast time steps. 

One possible solution for this is identify the ranks for the keys we want to read and create a loop over these ranks. 
Let's use bufr_dump to determine the number of timePeriods ( steps) by using

       bufr_dump -p bufr_khanun.bufr |grep "timePeriod"


To identify the ranks for the latitude, longitude we can use the following 

       bufr_dump -p bufr_khanun.bufr |grep -A3 "timePeriod"

This will find the timePeriod keys and show the next 3 lines after the match. 

#### Get and print forecasted values of the latitude,longitude and pressure

In [ ]:
#set number of forecast steps to 36 
NREP=36 

# initialise arrays
latp=np.zeros(NREP+1,dtype=float)
lonp=np.zeros(NREP+1,dtype=float)
pressure=np.zeros(NREP+1,dtype=float)

# loop over 36 forecast steps by calculating the rank for latitude, lomgitude and pressure
for k in range(1,NREP+1):
    rank=2*(k+1)  
    
    key=f"#{k}#timePeriod"
    timeStep=codes_get(bid,key)
        
    key=f"#{rank}#latitude"
    latp[k]=codes_get(bid,key)
    
    key=f"#{rank}#longitude"
    lonp[k]=codes_get(bid,key)
    
    key=f"#{k+1}#pressureReducedToMeanSeaLevel"
    pressure[k]=codes_get(bid,key)
    
    print(f" timePeriod: {timeStep} latitude : {latp[k]:.2f} longitude :{lonp[k]:.2f} pressure: {pressure[k]:9.2f}")


# Practicals - Decoding 



 ### Practical 1: Decode location and pressure of the storm in the analysis 

In [44]:
rank=2

key_lat=f"#{rank}#latitude"
lat=codes_get(bid,key_lat)
print(f" eccode key {key_lat} value {lat}")

key_lon=f"#{rank}#longitude"
lon=codes_get(bid,key_lon)
print(f" eccode key: {key_lon} value: {lon}")

key_press=f"#{rank-1}#pressureReducedToMeanSeaLevel"
pressure=codes_get(bid,key_press)
print(f" eccode key: {key_press} value: {pressure}")

 eccode key #2#latitude value 26.2
 eccode key: #2#longitude value: 125.5
 eccode key: #1#pressureReducedToMeanSeaLevel value: 95300.0



 ### Practical 2: Decode location of maximum wind of the tropical cyclone in analysis

In [45]:
rank=3

key_lat=f"#{rank}#latitude"
lat=codes_get(bid,key_lat)
print(f" eccode key {key_lat} value {lat}")

key_lon=f"#{rank}#longitude"
lon=codes_get(bid,key_lon)
print(f" eccode key: {key_lon} value: {lon}")

key_wind=f"#{rank-2}#windSpeedAt10M"
wind=codes_get(bid,key_wind)
print(f" eccode key: {key_wind} value: {wind}")

 eccode key #3#latitude value 26.6
 eccode key: #3#longitude value: 126.10000000000001
 eccode key: #1#windSpeedAt10M value: 34.0


## Decode and Encode BUFR File with bufr_dump
Using bufr_dump - Dpython or -Epython

> bufr_dump -Dpython my.bufr >sample_decode.py  


> bufr_dump -Epython my.bufr >sample_encode.py